# **Tweets Sentiment Analysis**

##### The goal of this project is to provide a foundational approach to Natural Language Processing (NLP). It involves preprocessing text data (tweets), applying vectorization techniques such as **CountVectorizer** and **TF-IDF**, and training various machine learning classifiers. The ultimate objective is to achieve the highest possible accuracy in a multi-class classification task: predicting whether the sentiment of a tweet is positive, neutral, or negative.

<img src="https://media.geeksforgeeks.org/wp-content/cdn-uploads/20210722215846/sentiment-analysis.jpg"></img>

#### **Download dependencies**

In [1]:
import pandas as pd
import numpy as np
import nltk

from sklearn.linear_model import  LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from xgboost import XGBClassifier

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from nltk.tokenize import word_tokenize
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import words
from nltk.metrics.distance import edit_distance
from nltk.tokenize import word_tokenize
from functools import lru_cache
from nltk.corpus import stopwords
from gensim.models import Word2Vec

nltk.download('punkt')
nltk.download('wordnet')
nltk.download('words')
nltk.download('stopwords')

[nltk_data] Downloading package punkt to /home/aelidrys/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to /home/aelidrys/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package words to /home/aelidrys/nltk_data...
[nltk_data]   Package words is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /home/aelidrys/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

#### **Load data & Basic preprocessing**

In [2]:
# load data from csv files
negative = pd.read_csv("./data/processedNegative.csv", header=None)
neutral = pd.read_csv("./data/processedNeutral.csv", header=None)
positive = pd.read_csv("./data/processedPositive.csv", header=None)

# concatenate all tweets and sentiments in a single dataset
neg_df = negative.iloc[0].to_list()
neu_df = neutral.iloc[0].to_list()
pos_df = positive.iloc[0].to_list()
tweets = neg_df + neu_df + pos_df
sentiment = (
    ['negative'] * len(negative.columns) +
    ['neutral'] * len(neutral.columns) +
    ['positive'] * len(positive.columns)
)
print(f"tweets size: {len(tweets)} | sentiment size: {len(sentiment)}")

# Encode sentiment labels
label_encoder = LabelEncoder().fit(sentiment)
sentiment_encoded = label_encoder.transform(sentiment)
target = sentiment_encoded

# drop missing values
df = pd.DataFrame({'tweet': tweets, 'sentiment': target})
print(df.value_counts('sentiment'))
print(f"missing values: {df.isna().sum().sum()}")
df.dropna(subset=['tweet'], inplace=True)
print(f"missing values after dropping: {df.isna().sum().sum()}")

# Remove punctuation, single character, and extra whitespace
df['tweet'] = df['tweet'].astype(str)
df['tweet'] = (
    df['tweet']
    .str.lower()
    .str.replace(r'[^\w\s]', '', regex=True)
    .str.replace(r'\b\w\b', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

# Remove duplicates
print(df.shape)
df.drop_duplicates(subset=['tweet'], inplace=True)
print(df.shape)

tweets size: 3873 | sentiment size: 3873
sentiment
1    1570
2    1186
0    1117
Name: count, dtype: int64
missing values: 5
missing values after dropping: 0
(3868, 2)
(3411, 2)


## **`I` - Tokenizer**

#### **Vectorization function**

In [3]:
def vectorize_data(data, vectorizer):
    X_train, X_val, y_train, y_val = train_test_split(
        data['tweet'], 
        data['sentiment'], 
        test_size=0.2, 
        random_state=42, 
        stratify=data['sentiment']
    )

    # Vectorization with the given vectorizer
    X_train = vectorizer.fit_transform(X_train)
    X_val = vectorizer.transform(X_val)
    return X_train, X_val, y_train, y_val


#### **Trining with tree models (Logistic regression, Random forest, xgboost)**

In [4]:
# Evaluate model
def evaluate_model(model, X_val, y_val, report=False):
    y_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_pred)
    print(f'accuracy: {accuracy:.3f}')
    if report:
        print(classification_report(y_val, y_pred))

# Train model
def train_models(X_train, y_train, X_val, y_val, report=False):

    # Train using logistic regression
    model = LogisticRegression(max_iter=1000)
    model.fit(X_train, y_train)
    print("Logistic Regression")
    evaluate_model(model, X_val, y_val, report)
    print("-" * 30)

    # Train using random forest
    model = RandomForestClassifier(n_estimators=350)
    model.fit(X_train, y_train)
    print("Random Forest")
    evaluate_model(model, X_val, y_val, report)
    print("-" * 30)

    # Train using xgboost
    model = XGBClassifier(n_estimators=350)
    model.fit(X_train, y_train)
    print("XGBoost")
    evaluate_model(model, X_val, y_val, report)
    print("-" * 30)

### **1) - Vectorization (0 or 1 if exists)**

In [5]:
# Vectorization with CountVectorizer binary=True (0 or 1 if exists)
vectorizer = CountVectorizer(binary=True)
X_train, X_val, y_train, y_val = vectorize_data(df, vectorizer)

##### **Top-10 most similar pair of tweets**

In [6]:



def get_top_similar_samples(X, tweets, top_k=10):
    similarity_matrix = cosine_similarity(X)

    np.fill_diagonal(similarity_matrix, 0)

    # avoid duplicate comparisons
    similarity_matrix = np.triu(similarity_matrix)    
    
    # Sort values and return flat indices
    flat_indices = np.argpartition(similarity_matrix.flatten(), -10)[-top_k:]
    
    # flip indices for descending sort
    flat_indices = np.flip(flat_indices)

    row_indices, col_indices = np.unravel_index(flat_indices, similarity_matrix.shape)

    
    print("--- Top 10 Most Similar Tweet Pairs ---")
    for rank, (row, col) in enumerate(zip(row_indices, col_indices)):
        score = similarity_matrix[row, col]
        print(f"\nRank {rank + 1} (Score: {score:.4f})")
        print(f"Tweet A (Index {row}): {tweets.iloc[row]}")
        print(f"Tweet B (Index {col}): {tweets.iloc[col]}")


X = vectorizer.fit_transform(df['tweet'])
get_top_similar_samples(X, df['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9780)
Tweet A (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1
Tweet B (Index 887): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for us to cont1

Rank 2 (Score: 0.9770)
Tweet A (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1
Tweet B (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1

Rank 3 (Score: 0.9770)
Tweet A (Index 513): hi ashish we tried to call your number but got no response unhappy please share another suitable time and an alternate cont1
Tweet B (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number 

##### **Training with different algorithms**

In [7]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.871
------------------------------
Random Forest
accuracy: 0.824
------------------------------
XGBoost
accuracy: 0.873
------------------------------


### **2) - Vectorization (frequency)**

In [8]:
# Vectorization with CountVectorizer (frequency)
vectorizer = CountVectorizer()
X_train, X_val, y_train, y_val = vectorize_data(df, vectorizer)

##### **Top-10 most similar pair of tweets**

In [9]:
X = vectorizer.fit_transform(df['tweet'])
get_top_similar_samples(X, df['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9798)
Tweet A (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1
Tweet B (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1

Rank 2 (Score: 0.9701)
Tweet A (Index 2476): hey thanks for being top new followers this week much appreciated happy want this
Tweet B (Index 2855): hey thanks for being my top new followers this week much appreciated happy want this

Rank 3 (Score: 0.9656)
Tweet A (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1
Tweet B (Index 887): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for us to cont1

Rank 4 (Score: 0.9636)
Tweet A (Index 57): koalas are dying of thirst and 

##### **Training with different algorithms**

In [10]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.868
------------------------------
Random Forest
accuracy: 0.829
------------------------------
XGBoost
accuracy: 0.868
------------------------------


### **3) - Vectorization (TFIDF)**

In [11]:
# Vectorization TF-IDF (term frequency-inverse document frequency)
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train, X_val, y_train, y_val = vectorize_data(df, vectorizer)

##### **Top-10 most similar pair of tweets**

In [12]:
X = vectorizer.fit_transform(df['tweet'])
get_top_similar_samples(X, df['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9531)
Tweet A (Index 2633): for the recent follow much appreciated happy want this
Tweet B (Index 2876): thanks for the recent follow much appreciated happy want this

Rank 2 (Score: 0.9469)
Tweet A (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1
Tweet B (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1

Rank 3 (Score: 0.9378)
Tweet A (Index 2476): hey thanks for being top new followers this week much appreciated happy want this
Tweet B (Index 2480): hey thanks for being top new followers this week much appreciated happy

Rank 4 (Score: 0.9359)
Tweet A (Index 2465): share the love thanks for being top new followers this week happy want this
Tweet B (Index 2864): share the love thanks for being top new followers this week happy

Rank 5 (Score: 0.9316)

##### **Training with different algorithms**

In [13]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.878
------------------------------
Random Forest
accuracy: 0.817
------------------------------
XGBoost
accuracy: 0.842
------------------------------


## **`II` Stemming**

#### **Spell Correction**

In [14]:
VOCAB = set(w.lower() for w in words.words())

@lru_cache(maxsize=10000)
def correct_word_levenshtein(word):
    word_lower = word.lower()
    
    if word_lower in VOCAB or len(word_lower) <= 2 or not word_lower.isalpha():
        return word
        
    candidates = [
        w for w in VOCAB 
        if w[0] == word_lower[0] and abs(len(w) - len(word_lower)) <= 2
    ]
    if not candidates:
        return word
        
    best_match = min(candidates, key=lambda cand: edit_distance(word_lower, cand))
    
    if edit_distance(word_lower, best_match) <= 2:
        return best_match
        
    return word

#### **Apply Stemming preprocessing technique**

In [15]:
stemmer = PorterStemmer()
def process_stemming(text, correct_misspelled=False):
    
    tokens = word_tokenize(text)
    if correct_misspelled:
        tokens_ = [correct_word_levenshtein(token) for token in tokens]
    stemmed_words = [stemmer.stem(word) for word in tokens]
    
    return " ".join(stemmed_words)


# Apply Stemming to tweets
df_stemmed = pd.DataFrame()
df_stemmed['tweet'] = df['tweet'].apply(process_stemming)
df_stemmed['sentiment'] = df['sentiment']
print(df_stemmed[['tweet']].head())

print(df_stemmed.shape)
df_stemmed.drop_duplicates(subset=['tweet'], inplace=True)
print(df_stemmed.shape)

                                               tweet
0                how unhappi some dog like it though
1  talk to my over driver about where im goingh s...
2  doe anybodi know if the rand like to fall agai...
3                miss go to gig in liverpool unhappi
4            there isnt new riverdal tonight unhappi
(3411, 2)
(3410, 2)


### **1) - Vectorization (0 or 1 if exists)**

In [16]:
# Vectoraization (0 or 1 if exists)
vectorizer = CountVectorizer(binary=True)
X_train, X_val, y_train, y_val = vectorize_data(df_stemmed, vectorizer)

##### **Top-10 most similar pair of tweets**

In [17]:
X = vectorizer.fit_transform(df_stemmed['tweet'])
get_top_similar_samples(X, df_stemmed['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9780)
Tweet A (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1
Tweet B (Index 886): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for us to cont1

Rank 2 (Score: 0.9770)
Tweet A (Index 513): hi ashish we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern cont1
Tweet B (Index 685): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number cont1

Rank 3 (Score: 0.9770)
Tweet A (Index 685): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number cont1
Tweet B (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1

Rank 4 (Score: 0.9636)
Tweet A (Index 2861): stat for

##### **Training with different algorithms**

In [18]:
# Training with different algorithms (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.865
------------------------------
Random Forest
accuracy: 0.836
------------------------------
XGBoost
accuracy: 0.870
------------------------------


### **2) - Vectorization (frequency)**

In [19]:
# Vectorization (frequency)
vectorizer = CountVectorizer()
X_train, X_val, y_train, y_val = vectorize_data(df_stemmed, vectorizer)

##### **Top-10 most similar pair of tweets**

In [20]:
X = vectorizer.fit_transform(df_stemmed['tweet'])
get_top_similar_samples(X, df_stemmed['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9798)
Tweet A (Index 685): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number cont1
Tweet B (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1

Rank 2 (Score: 0.9701)
Tweet A (Index 2475): hey thank for be top new follow thi week much appreci happi want thi
Tweet B (Index 2854): hey thank for be my top new follow thi week much appreci happi want thi

Rank 3 (Score: 0.9656)
Tweet A (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1
Tweet B (Index 886): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for us to cont1

Rank 4 (Score: 0.9636)
Tweet A (Index 57): koala are die of thirst and it all becaus of us unhappi
Tweet B (Index 304): are die of thirst an

##### **Training with different algorithms**

In [21]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.865
------------------------------
Random Forest
accuracy: 0.843
------------------------------
XGBoost
accuracy: 0.872
------------------------------


### **3) - Vectorization (TFIDF)**

In [22]:
# Vectorization (TF-IDF) Term Frequency-inverse Document Frequency
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train, X_val, y_train, y_val = vectorize_data(df_stemmed, vectorizer)

##### **Top-10 most similar pair of tweets**

In [23]:
X = vectorizer.fit_transform(df_stemmed['tweet'])
get_top_similar_samples(X, df_stemmed['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9552)
Tweet A (Index 2632): for the recent follow much appreci happi want thi
Tweet B (Index 2875): thank for the recent follow much appreci happi want thi

Rank 2 (Score: 0.9445)
Tweet A (Index 685): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number cont1
Tweet B (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1

Rank 3 (Score: 0.9357)
Tweet A (Index 2475): hey thank for be top new follow thi week much appreci happi want thi
Tweet B (Index 2479): hey thank for be top new follow thi week much appreci happi

Rank 4 (Score: 0.9332)
Tweet A (Index 2464): share the love thank for be top new follow thi week happi want thi
Tweet B (Index 2863): share the love thank for be top new follow thi week happi

Rank 5 (Score: 0.9289)
Tweet A (Index 2516): thank for the recent follow happi to connect happi

##### **Training with different algorithms**

In [24]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.875
------------------------------
Random Forest
accuracy: 0.839
------------------------------
XGBoost
accuracy: 0.850
------------------------------


## **`III` Lemmatization** 

#### **Apply Lemmatization preprocessing technique**

In [25]:
lemmatizer = WordNetLemmatizer()
def process_lemmatization(text, correct_misspelled=False):
    
    tokens = word_tokenize(text)
    if correct_misspelled:
        tokens_ = [correct_word_levenshtein(token) for token in tokens]
    lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens]
    
    return " ".join(lemmatized_words)

# Apply Lemmatization to tweets
df_lemmatized = pd.DataFrame()
df_lemmatized['tweet'] = df['tweet'].apply(process_lemmatization)
df_lemmatized['sentiment'] = df['sentiment']

print(df_lemmatized.shape)
df_lemmatized.drop_duplicates(subset=['tweet'], inplace=True)
print(df_lemmatized.shape)

(3411, 2)
(3410, 2)


### **1) - Vectorization (0 or 1 if exists)**

In [26]:
# Vectoraization (0 or 1 if exists)
vectorizer = CountVectorizer(binary=True)
X_train, X_val, y_train, y_val = vectorize_data(df_lemmatized, vectorizer)

##### **Top-10 most similar pair of tweets**

In [27]:
X = vectorizer.fit_transform(df_lemmatized['tweet'])
get_top_similar_samples(X, df_lemmatized['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 1.0000)
Tweet A (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1
Tweet B (Index 886): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for u to cont1

Rank 2 (Score: 0.9770)
Tweet A (Index 513): hi ashish we tried to call your number but got no response unhappy please share another suitable time and an alternate cont1
Tweet B (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1

Rank 3 (Score: 0.9770)
Tweet A (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1
Tweet B (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for c

##### **Training with different algorithms**

In [28]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.871
------------------------------
Random Forest
accuracy: 0.836
------------------------------
XGBoost
accuracy: 0.872
------------------------------


### **2) - Vectorization (frequency)**

In [29]:
# Vectorization (frequency)
vectorizer = CountVectorizer()
X_train, X_val, y_train, y_val = vectorize_data(df_lemmatized, vectorizer)

##### **Top-10 most similar pair of tweets**

In [30]:
X = vectorizer.fit_transform(df_lemmatized['tweet'])
get_top_similar_samples(X, df_lemmatized['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9827)
Tweet A (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1
Tweet B (Index 886): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for u to cont1

Rank 2 (Score: 0.9798)
Tweet A (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1
Tweet B (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1

Rank 3 (Score: 0.9701)
Tweet A (Index 2475): hey thanks for being top new follower this week much appreciated happy want this
Tweet B (Index 2854): hey thanks for being my top new follower this week much appreciated happy want this

Rank 4 (Score: 0.9644)
Tweet A (Index 685): hi we tried to call your number b

##### **Training with different algorithms**

In [31]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.875
------------------------------
Random Forest
accuracy: 0.845
------------------------------
XGBoost
accuracy: 0.864
------------------------------


### **3) - Vectorization (TFIDF)**

In [32]:
# Vectorization (TF-IDF) Term Frequency-inverse Document Frequency
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train, X_val, y_train, y_val = vectorize_data(df_lemmatized, vectorizer)

##### **Top-10 most similar pair of tweets**

In [33]:
X = vectorizer.fit_transform(df_lemmatized['tweet'])
get_top_similar_samples(X, df_lemmatized['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9530)
Tweet A (Index 2632): for the recent follow much appreciated happy want this
Tweet B (Index 2875): thanks for the recent follow much appreciated happy want this

Rank 2 (Score: 0.9509)
Tweet A (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1
Tweet B (Index 886): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for u to cont1

Rank 3 (Score: 0.9458)
Tweet A (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1
Tweet B (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1

Rank 4 (Score: 0.9380)
Tweet A (Index 2475): hey thanks for being top new follower this week much appreciated happy want this

##### **Training with different algorithms**

In [34]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.883
------------------------------
Random Forest
accuracy: 0.828
------------------------------
XGBoost
accuracy: 0.846
------------------------------


## **`IV` Stemming + misspellings correction**

#### **Apply Stemming + misspelling correction**

In [35]:
# Apply Stemming + misspelling correction to tweets
df_stemmed_misspellings = pd.DataFrame()
df_stemmed_misspellings['tweet'] = df['tweet'].apply(process_stemming, correct_misspelled=True)
df_stemmed_misspellings['sentiment'] = df['sentiment']
print(df_stemmed_misspellings[['tweet']].head())

print(df_stemmed_misspellings.shape)
df_stemmed_misspellings.drop_duplicates(subset=['tweet'], inplace=True)
print(df_stemmed_misspellings.shape)

                                               tweet
0                how unhappi some dog like it though
1  talk to my over driver about where im goingh s...
2  doe anybodi know if the rand like to fall agai...
3                miss go to gig in liverpool unhappi
4            there isnt new riverdal tonight unhappi
(3411, 2)
(3410, 2)


### **1) - Vectorization (0 or 1 if exists)**

In [36]:
# Vectoraization (0 or 1 if exists)
vectorizer = CountVectorizer(binary=True)
X_train, X_val, y_train, y_val = vectorize_data(df_stemmed_misspellings, vectorizer)

##### **Top-10 most similar pair of tweets**

In [37]:
X_all = vectorizer.fit_transform(df_stemmed_misspellings['tweet'])
get_top_similar_samples(X_all, df_stemmed_misspellings['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9780)
Tweet A (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1
Tweet B (Index 886): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for us to cont1

Rank 2 (Score: 0.9770)
Tweet A (Index 513): hi ashish we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern cont1
Tweet B (Index 685): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number cont1

Rank 3 (Score: 0.9770)
Tweet A (Index 685): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number cont1
Tweet B (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1

Rank 4 (Score: 0.9636)
Tweet A (Index 2861): stat for

##### **Training with different algorithms**

In [38]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.865
------------------------------
Random Forest
accuracy: 0.852
------------------------------
XGBoost
accuracy: 0.870
------------------------------


### **2) - Vectorization (frequency)**

In [39]:
# Vectorization (frequency)
vectorizer = CountVectorizer()
X_train, X_val, y_train, y_val = vectorize_data(df_stemmed_misspellings, vectorizer)

##### **Top-10 most similar pair of tweets**

In [40]:
X_all = vectorizer.fit_transform(df_stemmed_misspellings['tweet'])
get_top_similar_samples(X_all, df_stemmed_misspellings['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9798)
Tweet A (Index 685): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number cont1
Tweet B (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1

Rank 2 (Score: 0.9701)
Tweet A (Index 2475): hey thank for be top new follow thi week much appreci happi want thi
Tweet B (Index 2854): hey thank for be my top new follow thi week much appreci happi want thi

Rank 3 (Score: 0.9656)
Tweet A (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1
Tweet B (Index 886): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for us to cont1

Rank 4 (Score: 0.9636)
Tweet A (Index 57): koala are die of thirst and it all becaus of us unhappi
Tweet B (Index 304): are die of thirst an

##### **Training with different algorithms**

In [41]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.865
------------------------------
Random Forest
accuracy: 0.848
------------------------------
XGBoost
accuracy: 0.872
------------------------------


### **3) - Vectorization (TFIDF)**

In [42]:
# Vectorization (TF-IDF) Term Frequency-inverse Document Frequency
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train, X_val, y_train, y_val = vectorize_data(df_stemmed_misspellings, vectorizer)

##### **Top-10 most similar pair of tweets**

In [43]:
X_all = vectorizer.fit_transform(df_stemmed_misspellings['tweet'])
get_top_similar_samples(X_all, df_stemmed_misspellings['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9552)
Tweet A (Index 2632): for the recent follow much appreci happi want thi
Tweet B (Index 2875): thank for the recent follow much appreci happi want thi

Rank 2 (Score: 0.9445)
Tweet A (Index 685): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number cont1
Tweet B (Index 766): hi we tri to call your number but got no respons unhappi pleas share anoth suitabl time and an altern number for cont1

Rank 3 (Score: 0.9357)
Tweet A (Index 2475): hey thank for be top new follow thi week much appreci happi want thi
Tweet B (Index 2479): hey thank for be top new follow thi week much appreci happi

Rank 4 (Score: 0.9332)
Tweet A (Index 2464): share the love thank for be top new follow thi week happi want thi
Tweet B (Index 2863): share the love thank for be top new follow thi week happi

Rank 5 (Score: 0.9289)
Tweet A (Index 2516): thank for the recent follow happi to connect happi

##### **Training with different algorithms**

In [44]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.875
------------------------------
Random Forest
accuracy: 0.828
------------------------------
XGBoost
accuracy: 0.850
------------------------------


## **`V` Lemmatization + misspellings correction**

#### **Apply Lemmatization + misspelling correction**

In [45]:
# Apply Lemmatization + misspelling correction to tweets
df_lemmatized_misspellings = pd.DataFrame()
df_lemmatized_misspellings['tweet'] = df['tweet'].apply(process_lemmatization, correct_misspelled=True)
df_lemmatized_misspellings['sentiment'] = df['sentiment']
print(df_lemmatized_misspellings[['tweet']].head())

print(df_lemmatized_misspellings.shape)
df_lemmatized_misspellings.drop_duplicates(subset=['tweet'], inplace=True)
print(df_lemmatized_misspellings.shape)

                                               tweet
0                how unhappy some dog like it though
1  talking to my over driver about where im going...
2  doe anybody know if the rand likely to fall ag...
3             miss going to gig in liverpool unhappy
4           there isnt new riverdale tonight unhappy
(3411, 2)
(3410, 2)


### **1) - Vectorization (0 or 1 if exists)**

In [46]:
# Vectoraization (0 or 1 if exists)
vectorizer = CountVectorizer(binary=True)
X_train, X_val, y_train, y_val = vectorize_data(df_lemmatized_misspellings, vectorizer)

##### **Top-10 most similar pair of tweets**

In [47]:
X_all = vectorizer.fit_transform(df_lemmatized_misspellings['tweet'])
get_top_similar_samples(X_all, df_lemmatized_misspellings['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 1.0000)
Tweet A (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1
Tweet B (Index 886): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for u to cont1

Rank 2 (Score: 0.9770)
Tweet A (Index 513): hi ashish we tried to call your number but got no response unhappy please share another suitable time and an alternate cont1
Tweet B (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1

Rank 3 (Score: 0.9770)
Tweet A (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1
Tweet B (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for c

##### **Training with different algorithms**

In [48]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.871
------------------------------
Random Forest
accuracy: 0.840
------------------------------
XGBoost
accuracy: 0.872
------------------------------


### **2) - Vectorization (frequency)**

In [49]:
# Vectorization (frequency)
vectorizer = CountVectorizer()
X_train, X_val, y_train, y_val = vectorize_data(df_lemmatized_misspellings, vectorizer)

##### **Top-10 most similar pair of tweets**

In [50]:
X_all = vectorizer.fit_transform(df_lemmatized_misspellings['tweet'])
get_top_similar_samples(X_all, df_lemmatized_misspellings['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9827)
Tweet A (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1
Tweet B (Index 886): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for u to cont1

Rank 2 (Score: 0.9798)
Tweet A (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1
Tweet B (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1

Rank 3 (Score: 0.9701)
Tweet A (Index 2475): hey thanks for being top new follower this week much appreciated happy want this
Tweet B (Index 2854): hey thanks for being my top new follower this week much appreciated happy want this

Rank 4 (Score: 0.9644)
Tweet A (Index 685): hi we tried to call your number b

##### **Training with different algorithms**

In [51]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.875
------------------------------
Random Forest
accuracy: 0.845
------------------------------
XGBoost
accuracy: 0.864
------------------------------


### **3) - Vectorization (TFIDF)**

In [52]:
# Vectorization (TF-IDF) Term Frequency-inverse Document Frequency
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train, X_val, y_train, y_val = vectorize_data(df_lemmatized_misspellings, vectorizer)

##### **Top-10 most similar pair of tweets**

In [53]:
X_all = vectorizer.fit_transform(df_lemmatized_misspellings['tweet'])
get_top_similar_samples(X_all, df_lemmatized_misspellings['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9530)
Tweet A (Index 2632): for the recent follow much appreciated happy want this
Tweet B (Index 2875): thanks for the recent follow much appreciated happy want this

Rank 2 (Score: 0.9509)
Tweet A (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1
Tweet B (Index 886): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for u to cont1

Rank 3 (Score: 0.9458)
Tweet A (Index 685): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number cont1
Tweet B (Index 766): hi we tried to call your number but got no response unhappy please share another suitable time and an alternate number for cont1

Rank 4 (Score: 0.9380)
Tweet A (Index 2475): hey thanks for being top new follower this week much appreciated happy want this

##### **Training with different algorithms**

In [54]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.883
------------------------------
Random Forest
accuracy: 0.824
------------------------------
XGBoost
accuracy: 0.846
------------------------------


## **`VI` Tokenized and Stopwords Removal**

#### **Apply Stopwords Removal technique**

In [55]:
stop_words = set(stopwords.words('english'))
def filter_stopwords(text, correct_misspellings=False):
    tokens = word_tokenize(text.lower())
    filtered_tokens = [word for word in tokens if word not in stop_words]
    if correct_misspellings:
        filtered_tokens = [correct_word_levenshtein(word) for word in filtered_tokens]
    return ' '.join(filtered_tokens)

In [56]:
df_stopworded = pd.DataFrame()
df_stopworded['tweet'] = df['tweet'].apply(filter_stopwords)
df_stopworded['sentiment'] = df['sentiment']
print(df_stopworded.shape)
df_stopworded.drop_duplicates(subset=['tweet'], inplace=True)
print(df_stopworded.shape)

(3411, 2)
(3356, 2)


### **1) - Vectorization (0 or 1 if exists)**

In [57]:
# Vectoraization (0 or 1 if exists)
vectorizer = CountVectorizer(binary=True)
X_train, X_val, y_train, y_val = vectorize_data(df_stopworded, vectorizer)

##### **Top-10 most similar pair of tweets**

In [58]:
X_all = vectorizer.fit_transform(df_stopworded['tweet'])
get_top_similar_samples(X_all, df_stopworded['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9661)
Tweet A (Index 508): hi ashish tried call number got response unhappy please share another suitable time alternate cont1
Tweet B (Index 677): hi tried call number got response unhappy please share another suitable time alternate number cont1

Rank 2 (Score: 0.9661)
Tweet A (Index 677): hi tried call number got response unhappy please share another suitable time alternate number cont1
Tweet B (Index 874): hi tried call number got response unhappy please share another suitable time alternate number us cont1

Rank 3 (Score: 0.9487)
Tweet A (Index 2442): hey thanks top new followers week much appreciated happy want
Tweet B (Index 2446): hey thanks top new followers week much appreciated happy

Rank 4 (Score: 0.9428)
Tweet A (Index 2912): thanks recent follow happy connect happy great thursday get
Tweet B (Index 3084): thanks recent follow happy connect happy great thursday get free

Rank 5 (Score: 0.9428)
Tweet A (Index 2766):

##### **Training with different algorithms**

In [59]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.872
------------------------------
Random Forest
accuracy: 0.804
------------------------------
XGBoost
accuracy: 0.866
------------------------------


### **2) - Vectorization (frequency)**

In [60]:
# Vectorization (frequency)
vectorizer = CountVectorizer()
X_train, X_val, y_train, y_val = vectorize_data(df_stopworded, vectorizer)

##### **Top-10 most similar pair of tweets**

In [61]:
X_all = vectorizer.fit_transform(df_stopworded['tweet'])
get_top_similar_samples(X_all, df_stopworded['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9718)
Tweet A (Index 677): hi tried call number got response unhappy please share another suitable time alternate number cont1
Tweet B (Index 874): hi tried call number got response unhappy please share another suitable time alternate number us cont1

Rank 2 (Score: 0.9574)
Tweet A (Index 2912): thanks recent follow happy connect happy great thursday get
Tweet B (Index 3084): thanks recent follow happy connect happy great thursday get free

Rank 3 (Score: 0.9574)
Tweet A (Index 2483): thanks recent follow happy connect happy great thursday want
Tweet B (Index 2960): thanks recent follow happy connect happy great thursday want free

Rank 4 (Score: 0.9535)
Tweet A (Index 3311): thanks recent follow happy connect happy great wednesday
Tweet B (Index 3341): thanks recent follow happy connect happy great wednesday want

Rank 5 (Score: 0.9535)
Tweet A (Index 2466): thanks recent follow happy connect happy great thursday
Tweet B (Index

##### **Training with different algorithms**

In [62]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.874
------------------------------
Random Forest
accuracy: 0.798
------------------------------
XGBoost
accuracy: 0.868
------------------------------


### **3) - Vectorization (TFIDF)**

In [63]:
# Vectorization (TF-IDF) Term Frequency-inverse Document Frequency
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
X_train, X_val, y_train, y_val = vectorize_data(df_stopworded, vectorizer)

##### **Top-10 most similar pair of tweets**

In [64]:
X_all = vectorizer.fit_transform(df_stopworded['tweet'])
get_top_similar_samples(X_all, df_stopworded['tweet'], 10)

--- Top 10 Most Similar Tweet Pairs ---

Rank 1 (Score: 0.9566)
Tweet A (Index 2442): hey thanks top new followers week much appreciated happy want
Tweet B (Index 2446): hey thanks top new followers week much appreciated happy

Rank 2 (Score: 0.9505)
Tweet A (Index 2431): share love thanks top new followers week happy want
Tweet B (Index 2825): share love thanks top new followers week happy

Rank 3 (Score: 0.9373)
Tweet A (Index 2983): great thursday looking forward reading tweets happy want
Tweet B (Index 2992): great thursday looking forward reading tweets happy want free

Rank 4 (Score: 0.9354)
Tweet A (Index 2483): thanks recent follow happy connect happy great thursday want
Tweet B (Index 2960): thanks recent follow happy connect happy great thursday want free

Rank 5 (Score: 0.9317)
Tweet A (Index 2466): thanks recent follow happy connect happy great thursday
Tweet B (Index 2483): thanks recent follow happy connect happy great thursday want

Rank 6 (Score: 0.9308)
Tweet A (Index 

##### **Training with different algorithms**

In [65]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.876
------------------------------
Random Forest
accuracy: 0.763
------------------------------
XGBoost
accuracy: 0.850
------------------------------


## **`VII` Word2vec**

#### **Apply word2vec**

In [66]:

data = df_lemmatized_misspellings
tokenized_tweets = [word_tokenize(tweet) for tweet in data['tweet']]

def get_tweet_vector(tokens, model):
    vectors = [model.wv[word] for word in tokens if word in model.wv]
    if len(vectors) > 0:
        return np.mean(vectors, axis=0)
    else:
        return np.zeros(model.vector_size)

X_train, X_val, y_train, y_val = train_test_split(
    tokenized_tweets,
    data['sentiment'],
    test_size=0.2,
    random_state=42,
    stratify=data['sentiment']
)

model1 = Word2Vec(sentences=X_train, vector_size=200, epochs=1000, min_count=1, window=5, workers=4, alpha=0.01)

X_train = np.array([get_tweet_vector(tokens, model1) for tokens in X_train])
X_val = np.array([get_tweet_vector(tokens, model1) for tokens in X_val])

print((X_train.shape))
print((X_val.shape))

(2728, 200)
(682, 200)


#### **Training with different algorithms**

In [67]:
# Training with different models (logistic regression, random forest, xgboost)
train_models(X_train, y_train, X_val, y_val)

Logistic Regression
accuracy: 0.853
------------------------------
Random Forest
accuracy: 0.817
------------------------------
XGBoost
accuracy: 0.821
------------------------------
